In [1]:
import os
import pandas as pd
import geopandas as gpd
from matplotlib import pyplot as plt
from glob import glob
import numpy as np
from spectral.io import envi
from tqdm import tqdm

os.chdir('/store/carroll/sbgplants/')

In [4]:
# file paths
raw = 'data/raw'
rdn_fol = '/store/carroll/col/data/2018/raw/L1/'
rfl_fol = '/store/carroll/col/data/2018/deploy_3c_20251001/'
out_folder = 'data/out_csv'

table = 'extracted_spectra'

In [3]:
# load relevant output tables

pixel = pd.read_csv(os.path.join(out_folder, 'pixel.csv'))
fids = pixel.granule_id.unique()

In [28]:
# extract rdn, rfl, unc per px

fps_rdn = [x for x in glob(os.path.join(rdn_fol, '*/*rdn_ort.hdr')) if any(xx in x for xx in fids)]
fps_rfl = [x for x in glob(os.path.join(rfl_fol, '*/output/*rfl.hdr')) if any(xx in x for xx in fids) and 'subs' not in x]
fps_unc = [x.replace('rfl','uncert') for x in fps_rfl]

id_cols = pixel.columns
val_cols = list(range(int(envi.read_envi_header(fps_rdn[0])['bands'])))

# rdn
out = []
for fp in tqdm(fps_rdn):
    fid = fp.split('/')[-1].removesuffix('_rdn_ort.hdr')
    tmp = pixel[pixel['granule_id']==fid].copy()
    # extract spectra
    rdn = envi.open(fp).open_memmap()
    r = tmp['glt_row']; c=tmp['glt_column']
    vals = rdn[r, c, :]
    # format df
    tmp = pd.concat([tmp, pd.DataFrame(vals, index=tmp.index, columns=val_cols)], axis=1)
    tmp = tmp.melt(id_vars=id_cols, value_vars=val_cols, var_name='band_number', value_name='radiance')
    out.append(tmp)
df_rdn = pd.concat(out)

# rfl
out = []
for fp in tqdm(fps_rfl):
    fid = fp.split('/')[-1].removesuffix('_rfl.hdr')
    tmp = pixel[pixel['granule_id']==fid].copy()
    # extract spectra
    rfl = envi.open(fp).open_memmap()
    r = tmp['glt_row']; c=tmp['glt_column']
    vals = rfl[r, c, :]
    # format df
    tmp = pd.concat([tmp, pd.DataFrame(vals, index=tmp.index, columns=val_cols)], axis=1)
    tmp = tmp.melt(id_vars=id_cols, value_vars=val_cols, var_name='band_number', value_name='reflectance')
    out.append(tmp)
df_rfl = pd.concat(out)

# unc
out = []
for fp in tqdm(fps_unc):
    fid = fp.split('/')[-1].removesuffix('_uncert.hdr')
    tmp = pixel[pixel['granule_id']==fid].copy()
    # extract spectra
    unc = envi.open(fp).open_memmap()
    r = tmp['glt_row']; c=tmp['glt_column']
    vals = unc[r, c, :]
    # format df
    tmp = pd.concat([tmp, pd.DataFrame(vals, index=tmp.index, columns=val_cols)], axis=1)
    tmp = tmp.melt(id_vars=id_cols, value_vars=val_cols, var_name='band_number', value_name='uncertainty_ref')
    out.append(tmp)
df_unc = pd.concat(out)

100%|██████████████████████████████████████████████████████████████████████████████████████████| 61/61 [02:10<00:00,  2.14s/it]


In [37]:
out_table = (
    df_rdn
    .merge(df_rfl, on=['pixel_id','band_number'], how='inner')
    .merge(df_unc, on=['pixel_id','band_number'], how='inner')
)

out_table['extract_id'] = range(len(out_table))

out_table = out_table[['extract_id', 'pixel_id', 'band_number', 'radiance', 'reflectance', 'uncertainty_ref']]

In [38]:
out_table

,extract_id,pixel_id,band_number,radiance,reflectance,uncertainty_ref
0,0,1303,0,2.040740,-0.01,0.026072
1,1,1304,0,1.962031,-0.01,0.021608
2,2,1305,0,1.906291,-0.01,0.027805
3,3,1306,0,2.055831,-0.01,0.026777
4,4,1303,1,1.702304,-0.01,0.027024
...,...,...,...,...,...,...
6490531,6490531,3799,425,0.025766,-0.01,0.033603
6490532,6490532,3800,425,0.023561,-0.01,0.025578
6490533,6490533,3801,425,0.019022,-0.01,0.023105
6490534,6490534,3802,425,0.023032,-0.01,0.024512


In [39]:
# export table
fp_out = os.path.join(out_folder, f'{table}.csv')
out_table.to_csv(fp_out, index=False)